In [ ]:
import os
import shutil

# Directory to start searching from
root_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r32a64_ranknet0/"

# Walk through the directory and all subdirectories recursively
for root, dirs, files in os.walk(root_dir, topdown=False):
    for dir_name in dirs:
        # Check if the directory ends with '.merged'
        if dir_name.endswith("_merged"):
            dir_path = os.path.join(root, dir_name)
            try:
                # Delete the directory and all its contents
                shutil.rmtree(dir_path)
                print(f"Deleted directory: {dir_path}")
            except Exception as e:
                print(f"Error deleting {dir_path}: {e}")


In [5]:
from itertools import product
import torch
from torch.nn import BCEWithLogitsLoss
def rankNet(y_pred, y_true, padded_value_indicator=-1, weight_by_diff=False, weight_by_diff_powed=False):
    """
    RankNet loss introduced in "Learning to Rank using Gradient Descent".
    :param y_pred: predictions from the model, shape [batch_size, slate_length]
    :param y_true: ground truth labels, shape [batch_size, slate_length]
    :param weight_by_diff: flag indicating whether to weight the score differences by ground truth differences.
    :param weight_by_diff_powed: flag indicating whether to weight the score differences by the squared ground truth differences.
    :return: loss value, a torch.Tensor
    """
    y_pred = y_pred.clone()
    y_true = y_true.clone()

    mask = y_true == padded_value_indicator
    y_pred[mask] = float('-inf')
    y_true[mask] = float('-inf')

    # here we generate every pair of indices from the range of document length in the batch
    document_pairs_candidates = list(product(range(y_true.shape[1]), repeat=2))

    pairs_true = y_true[:, document_pairs_candidates]
    selected_pred = y_pred[:, document_pairs_candidates]

    # here we calculate the relative true relevance of every candidate pair
    true_diffs = pairs_true[:, :, 0] - pairs_true[:, :, 1]
    pred_diffs = selected_pred[:, :, 0] - selected_pred[:, :, 1]

    # here we filter just the pairs that are 'positive' and did not involve a padded instance
    # we can do that since in the candidate pairs we had symetric pairs so we can stick with
    # positive ones for a simpler loss function formulation
    the_mask = (true_diffs > 0) & (~torch.isinf(true_diffs))

    pred_diffs = pred_diffs[the_mask]

    weight = None
    if weight_by_diff:
        abs_diff = torch.abs(true_diffs)
        weight = abs_diff[the_mask]
    elif weight_by_diff_powed:
        true_pow_diffs = torch.pow(pairs_true[:, :, 0], 2) - torch.pow(pairs_true[:, :, 1], 2)
        abs_diff = torch.abs(true_pow_diffs)
        weight = abs_diff[the_mask]

    # here we 'binarize' true relevancy diffs since for a pairwise loss we just need to know
    # whether one document is better than the other and not about the actual difference in
    # their relevancy levels
    true_diffs = (true_diffs > 0).type(torch.float32)
    true_diffs = true_diffs[the_mask]

    return BCEWithLogitsLoss(weight=weight)(pred_diffs, true_diffs)

In [21]:
print(rankNet(torch.Tensor([[1,2,3,4]]),torch.Tensor([[4,3,2,1]])))
print(rankNet(torch.Tensor([[1,2,3,3]]),torch.Tensor([[3,3,3,3]])))

tensor(1.8737)
tensor(nan)


# 使用vllm生成

In [46]:
import torch
import gc

del llm
gc.collect()
torch.cuda.empty_cache()
# Reset CUDA device to fully clear memory
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()  # Wait for all str

In [1]:
from vllm import LLM
import torch
from unsloth import FastLanguageModel
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r8a16_client1/20"
model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-30"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-10"
# model, tokenizer = FastLanguageModel.from_pretrained(model_cache_dir, load_in_4bit=True, dtype=None)
# model.save_pretrained_merged(f"{model_cache_dir}_merged", tokenizer, save_method = "merged_16bit",)
llm = LLM(model=f"{model_cache_dir}_merged", tensor_parallel_size=1, dtype=torch.bfloat16, trust_remote_code=True, 
    enable_lora=False, max_model_len=2048, gpu_memory_utilization=0.8)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[2025-01-07 06:02:52,588] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/tiger/.pyenv/versions/3.11.2/lib/python3.11/site-packages/bytedmetrics/__init__.py:10: UserWarning: bytedmetrics is renamed to bytedance.metrics, please using `bytedance.metrics` instead of `bytedmetrics`
  warnings.warn("bytedmetrics is renamed to bytedance.metrics, please using `bytedance.metrics` instead of `bytedmetrics`")
/usr/bin/ld: cannot find -laio
collect2: error: ld returned 1 exit status
/usr/bin/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dlvsym'
/usr/bin/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dlopen'
/usr/bin/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dlclose'
/usr/bin/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dlerror'
/usr/bin/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dlsym'
collect2: error: ld returned 1 exit status


INFO 01-07 06:03:01 config.py:510] This model supports multiple tasks: {'score', 'reward', 'classify', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 01-07 06:03:01 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-30_merged', speculative_config=None, tokenizer='/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-30_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(

[W107 06:03:02.394913513 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 01-07 06:03:06 model_runner.py:1099] Loading model weights took 14.9712 GB
INFO 01-07 06:03:07 worker.py:241] Memory profiling takes 0.79 seconds
INFO 01-07 06:03:07 worker.py:241] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.80) = 63.32GiB
INFO 01-07 06:03:07 worker.py:241] model weights take 14.97GiB; non_torch_memory takes 0.11GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 47.05GiB.
INFO 01-07 06:03:07 gpu_executor.py:76] # GPU blocks: 24088, # CPU blocks: 2048
INFO 01-07 06:03:07 gpu_executor.py:80] Maximum concurrency for 2048 tokens per request: 188.19x
INFO 01-07 06:03:09 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_util

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:28<00:00,  1.21it/s]

INFO 01-07 06:03:38 model_runner.py:1535] Graph capturing finished in 29 secs, took 0.25 GiB
INFO 01-07 06:03:38 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 31.71 seconds


In [3]:
import datasets
eval_set = datasets.load_dataset("json", data_files="alpaca_eval_output/eval_set_256.json")["train"]
len(eval_set)

Generating train split: 256 examples [00:00, 50614.77 examples/s]


256

In [ ]:
template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{} 

### Response:"""

def format_input(row):
    row["instruction"] = template.format(row["instruction"])
    return row

eval_set = eval_set.map(format_input)
print(eval_set[:3])

In [3]:
# import datasets
# from tqdm import tqdm
# import random

# eval_set = datasets.load_dataset("tatsu-lab/alpaca_eval", "alpaca_eval")["eval"]
# eval_set_256 = eval_set.select(random.sample(range(len(eval_set)), k=256))
# eval_set_256.to_json("alpaca_eval_output/eval_set_256.json")

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 255.84ba/s]


196501

In [5]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256
)

outputs = llm.generate(eval_set["instruction"], sampling_params)

Processed prompts:   0%|          | 0/256 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 256/256 [00:10<00:00, 23.97it/s, est. speed input: 1493.55 toks/s, output: 4079.32 toks/s]


In [5]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256
)

In [17]:
template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{} 

### Response:"""

QA_template = """The following are multiple choice questions (with answers).\n\n Question: {question}\nChoose from 4 options.\nOptions:\n{choices}\nAnswer:"""

outputs = llm.generate(QA_template.format_map({"question":"""Which piece of safety equipment is used to keep mold spores from entering the respiratory system?""", "choices":"""A: safety goggles.\n\n B: breathing mask.\n\n C: rubber gloves.\n\n D: lead apron."""}), sampling_params)

print(outputs[0].outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 23.25it/s, est. speed input: 1447.45 toks/s, output: 46.67 toks/s]

 B


In [6]:
def write_result(row, idx):
    row["output"] = outputs[idx].outputs[0].text
    row["generator"] = "c10s2round30FedSPA"
    return row

In [7]:
eval_set = eval_set.map(write_result, with_indices=True)

Map: 100%|██████████| 256/256 [00:00<00:00, 20501.04 examples/s]


In [8]:
eval_set.to_json("alpaca_eval_output/c10s2round30FedSPA_256.json")

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 289.94ba/s]


306151

In [32]:
import pandas as pd
data = pd.read_json("alpaca_eval_output/3_3_c1_20r.json", lines=True)

# 评估preference consistency

In [1]:
from src.utils.prompts import judge_prompt

In [2]:
import datasets
eval_set = datasets.load_dataset("json", data_files="/mnt/bn/data-tns-live-llm/leon/FedSPA/alpaca_eval_output/M0_256.json")["train"]
len(eval_set)

/home/tiger/miniconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 256 examples [00:00, 37326.77 examples/s]


256

In [3]:
eval_set = eval_set.to_pandas()
eval_set.head()

,instruction,output,generator,dataset
0,Below is an instruction that describes a task....,Here is a medium-level Sudoku puzzle. This puz...,M0,selfinstruct
1,Below is an instruction that describes a task....,This email subject line contains information t...,M0,selfinstruct
2,Below is an instruction that describes a task....,"Microsoft, Google, Nintendo, Sony, EA. The ord...",M0,koala
3,Below is an instruction that describes a task....,"As of 2023, the largest ocean in the world is ...",M0,oasst
4,Below is an instruction that describes a task....,A Giant Spider Blocks Your Path.,M0,selfinstruct


# 每个 instruction 生成 4 条 response

In [10]:
prompts = []
for instruction in eval_set["instruction"]:
    prompts.extend([instruction]*4)
print(len(prompts))

1024


In [11]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256
)

outputs = llm.generate(prompts, sampling_params)

Processed prompts:   0%|          | 0/1024 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1024/1024 [00:45<00:00, 22.30it/s, est. speed input: 1389.93 toks/s, output: 3976.73 toks/s]


In [12]:
template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{} 

### Response:"""

In [13]:
judge_prompts = []
for prompt, output in zip(prompts, outputs):
    formatted_llm_prompt = judge_prompt.format(
        prompt=prompt, response=output.outputs[0].text
    )

    judge_prompts.append(template.format(formatted_llm_prompt))

In [22]:
import json
json.dump(judge_prompts, open("judge_prompts_tmp.json", "w"))

# 模型本身进行judge

In [1]:
import json
judge_prompts = json.load(open("judge_prompts_tmp.json","r"))
judge_prompts[:5]

['Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nReview the user’s question and the corresponding response using the additive 5-point\nscoring system described below. \n\nThe user\'s question is between <question> and </question>\nThe response of the AI Assistant is between <response> and </response>\n\nPoints are accumulated based on the satisfaction of each\ncriterion:\n- Add 1 point if the response is relevant and provides some information related to\nthe user’s inquiry, even if it is incomplete or contains some irrelevant content.\n- Add another point if the response addresses a substantial portion of the user’s question,\nbut does not completely resolve the query or provide a direct answer.\n- Award a third point if the response answers the basic elements of the user’s question in a\nuseful way, regardless of whether it seems to have been written by an AI Assistant or if it\nhas elements typically foun

In [5]:
import torch
import gc

# del llm
gc.collect()
torch.cuda.empty_cache()
# Reset CUDA device to fully clear memory
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()  # Wait for all str

from vllm import LLM
import torch
from unsloth import FastLanguageModel
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r8a16_client0/20"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/M0"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/29/8/2"
model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-29"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r32a64_ranknet0/20/1/2"
model, tokenizer = FastLanguageModel.from_pretrained(model_cache_dir, load_in_4bit=True, dtype=None)
model.save_pretrained_merged(f"{model_cache_dir}_merged", tokenizer, save_method = "merged_16bit",)
llm = LLM(model=f"{model_cache_dir}_merged", tensor_parallel_size=1, dtype=torch.bfloat16, trust_remote_code=True, 
    enable_lora=False, max_model_len=2048, gpu_memory_utilization=0.8)

Exception ignored in: <function LLM.__del__ at 0x7f2d58d074c0>
Traceback (most recent call last):
  File "/home/tiger/miniconda3/envs/llm/lib/python3.11/site-packages/vllm/entrypoints/llm.py", line 236, in __del__
    if self.llm_engine and hasattr(self.llm_engine, "shutdown"):
       ^^^^^^^^^^^^^^^
AttributeError: 'LLM' object has no attribute 'llm_engine'


==((====))==  Unsloth 2024.12.8: Fast Llama patching. Transformers: 4.47.1.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1. CUDA: 8.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2024.12.8 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 1434.97 out of 2015.23 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 32/32 [00:00<00:00, 102.18it/s]


Unsloth: Saving tokenizer... Done.
Done.
INFO 12-24 19:16:25 config.py:478] This model supports multiple tasks: {'reward', 'classify', 'generate', 'embed', 'score'}. Defaulting to 'generate'.
INFO 12-24 19:16:25 llm_engine.py:249] Initializing an LLM engine (v0.6.5) with config: model='/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-29_merged', speculative_config=None, tokenizer='/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-29_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cu

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:11<00:34, 11.60s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:23<00:23, 11.59s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:35<00:11, 11.80s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:38<00:00,  8.26s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:38<00:00,  9.52s/it]



INFO 12-24 19:17:05 model_runner.py:1097] Loading model weights took 14.9595 GB
INFO 12-24 19:17:06 worker.py:241] Memory profiling takes 0.79 seconds
INFO 12-24 19:17:06 worker.py:241] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.80) = 63.32GiB
INFO 12-24 19:17:06 worker.py:241] model weights take 14.96GiB; non_torch_memory takes 0.01GiB; PyTorch activation peak memory takes 1.18GiB; the rest of the memory reserved for KV Cache is 47.18GiB.
INFO 12-24 19:17:06 gpu_executor.py:76] # GPU blocks: 24154, # CPU blocks: 2048
INFO 12-24 19:17:06 gpu_executor.py:80] Maximum concurrency for 2048 tokens per request: 188.70x
INFO 12-24 19:17:07 model_runner.py:1413] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 12-24 19:17:07 model_runner.py:1417] If out-of-memory error occurs during cudagraph cap

In [6]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256
)
judge_outputs = llm.generate(judge_prompts, sampling_params)

from src.utils.generate_scores import extract_scores
grouped_outputs = []
for i in range(0, len(judge_outputs), 4):
    group = [extract_scores(e.outputs[0].text) for e in judge_outputs[i:i+4]]
    grouped_outputs.append(group)
print(grouped_outputs[0])

Processed prompts: 100%|██████████| 1024/1024 [01:09<00:00, 14.69it/s, est. speed input: 9344.73 toks/s, output: 1495.54 toks/s] 


[0, 5, 0, 5]


In [92]:
print(judge_outputs[0].outputs[0].text)

score: 0

The response fails to complete the task of designing a medium-level Sudoku puzzle. It starts by providing a partial puzzle but does not complete it, nor does it explain how to solve it or check its validity. The response lacks a clear structure, explanation, or any additional information that would make it a complete and useful solution to the user's request. The response is also not concise and to the point, as it introduces unnecessary content unrelated to the task.


In [23]:
import json
json.dump(grouped_outputs, open("grouped_outputs_tmp.json", "w"))

In [ ]:
from pprint import pprint as pp
pp(judge_prompts[:4])

# 另一个模型进行judge

In [16]:
import torch
import gc

del llm
gc.collect()
torch.cuda.empty_cache()
# Reset CUDA device to fully clear memory
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()  # Wait for all str
from vllm import LLM
import torch
from unsloth import FastLanguageModel
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r32a64_ranknet0/20/0/2"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/29/1/2"
model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/29/1/2"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/M0"
# model, tokenizer = FastLanguageModel.from_pretrained(model_cache_dir, load_in_4bit=True, dtype=None)
# model.save_pretrained_merged(f"{model_cache_dir}_merged", tokenizer, save_method = "merged_16bit",)
llm = LLM(model=f"{model_cache_dir}_merged", tensor_parallel_size=1, dtype=torch.bfloat16, trust_remote_code=True, 
    enable_lora=False, max_model_len=2048, gpu_memory_utilization=0.8)

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:11<00:33, 11.16s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:22<00:22, 11.25s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:34<00:11, 11.46s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:37<00:00,  8.05s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:37<00:00,  9.25s/it]



In [17]:
judge_outputs_1 = llm.generate(judge_prompts, sampling_params)
from src.utils.generate_scores import extract_scores
grouped_outputs_1 = []
for i in range(0, len(judge_outputs_1), 4):
    group = [extract_scores(e.outputs[0].text) for e in judge_outputs_1[i:i+4]]
    grouped_outputs_1.append(group)
print(grouped_outputs_1[0])

Processed prompts: 100%|██████████| 1024/1024 [01:10<00:00, 14.43it/s, est. speed input: 9182.28 toks/s, output: 1477.89 toks/s] 


[0, 5, 0, 5]


# 使用 GPT4o 进行 judge

In [2]:
import json
judge_prompts = json.load(open("judge_prompts_tmp.json", "r"))
grouped_outputs = json.load(open("grouped_outputs_tmp.json", "r"))

In [1]:
import openai
client = openai.AzureOpenAI(
    azure_endpoint="https://gpt-i18n.byteintl.net/gpt/openapi/online/v2/crawl",
    api_version="2024-08-06",
    api_key="uO3ZsKN6On8H3aY8ZAcupOaq2KAuTMWJ"
)

In [11]:
import openai
from tqdm import tqdm
from multiprocessing import Pool

# 使用 OpenAI GPT-4o 进行生成
def generate_with_openai(prompt):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=messages,
        max_tokens=256,
        temperature=0.7,
        top_p=0.9
    )
    return response.choices[0].message.content

# 使用多进程进行生成
if __name__ == '__main__':
    with Pool(8) as pool:
        judge_outputs_1 = list(tqdm(pool.imap(generate_with_openai, judge_prompts), total=len(judge_prompts)))

    from src.utils.generate_scores import extract_scores
    grouped_outputs_1 = []
    for i in range(0, len(judge_outputs_1), 4):
        group = [extract_scores(output) for output in judge_outputs_1[i:i+4]]
        grouped_outputs_1.append(group)
    print(grouped_outputs_1[0])

  0%|          | 0/1024 [10:54<?, ?it/s]
/home/tiger/miniconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-12-14 11:38:34,796	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


[0, 0, 0, 0]


In [18]:
json.dump(grouped_outputs_1, open("gpt4o_grouped_outputs.json", "w"))

In [9]:
def scores_to_ranks(scores):
    """
    将分数转换为排名，相同分数按照出现顺序排名
    例如: [3,3,1,3] -> [1,2,4,3]
    """
    # 过滤掉None值
    valid_scores = [(score, idx) for idx, score in enumerate(scores) if score is not None]
    if not valid_scores:
        return [None] * len(scores)
    
    # 按分数降序排序，保持原始索引
    sorted_scores = sorted(valid_scores, key=lambda x: (x[0], -x[1]), reverse=True)
    
    # 生成排名（1开始）
    ranks = [None] * len(scores)
    for rank, (_, original_idx) in enumerate(sorted_scores, 1):
        ranks[original_idx] = rank
        
    return ranks

In [37]:
scores_to_ranks([1,1,1,1])

[1, 2, 3, 4]

In [18]:
# 转换两组输出的排名
grouped_ranks_0 = []
grouped_ranks_1 = []

for group0, group1 in zip(grouped_outputs, grouped_outputs_1):
    ranks0 = scores_to_ranks(group0)
    ranks1 = scores_to_ranks(group1)
    grouped_ranks_0.append(ranks0)
    grouped_ranks_1.append(ranks1)

# 打印示例看看转换效果
print("原始分数示例:")
print(f"组1: {grouped_outputs[0]}")
print(f"组2: {grouped_outputs_1[0]}")
print("\n转换后的排名:")
print(f"组1: {grouped_ranks_0[0]}")
print(f"组2: {grouped_ranks_1[0]}")

原始分数示例:
组1: [0, 5, 0, 5]
组2: [0, 5, 0, 5]

转换后的排名:
组1: [3, 1, 4, 2]
组2: [3, 1, 4, 2]


In [19]:
from scipy import stats
import numpy as np

# 计算每组内的一致性
group_agreements = []
for group0, group1 in zip(grouped_ranks_0, grouped_ranks_1):
    # 检查每组4个评分是否一致
    agreement, _ = stats.kendalltau(group0, group1)
    if not np.isnan(agreement):  # 只添加有效的相关系数
        group_agreements.append(agreement)
    else:
        print(group0, group1)
        group_agreements.append(1)

# print(group_agreements)
print(f"平均组内一致性: {np.mean(group_agreements):.4f}")

平均组内一致性: 0.6380


In [186]:
print(f"组1: {grouped_outputs[:100]}")
print(f"组2: {grouped_outputs_1[:100]}")

组1: [[0, 5, 0, 1], [1, 2, 1, 5], [1, 1, 3, 1], [4, 5, 5, 5], [1, 1, 1, 1], [5, 5, 5, 3], [0, 3, 2, 5], [5, 4, 4, 3], [2, 3, 0, 4], [1, 2, 1, 2], [2, 1, 3, 1], [4, 5, 5, 5], [5, 1, 4, 4], [2, 2, 1, 3], [5, 3, 1, 5], [3, 4, 3, 2], [5, 4, 4, 5], [2, 3, 4, 3], [4, 4, 5, 4], [0, 3, 1, 3], [2, 2, 3, 4], [3, 3, 3, 1], [2, 5, 5, 4], [2, 2, 1, 2], [4, 5, 1, 1], [5, 0, 1, 5], [1, 3, 1, 2], [3, 4, 3, 5], [1, 4, 5, 3], [0, 2, 5, 4], [2, 1, 0, 0], [3, 1, 2, 1], [3, 4, 1, 4], [2, 3, 3, 5], [3, 2, 1, 4], [4, 5, 1, 1], [2, 4, 0, 4], [4, 2, 3, 2], [4, 1, 5, 5], [3, 4, 2, 3], [3, 1, 2, 3], [5, 5, 4, 3], [0, 3, 0, 0], [2, 3, 2, 4], [5, 1, 3, 1], [1, 1, 2, 2], [4, 5, 2, 3], [3, 3, 5, 3], [1, 4, 5, 5], [5, 5, 5, 5], [4, 5, 4, 5], [2, 5, 0, 0], [0, 0, 1, 1], [3, 3, 1, 4], [0, 4, 3, 1], [5, 1, 1, 1], [0, 0, 0, 0], [1, 1, 3, 3], [3, 1, 1, 1], [1, 4, 5, 5], [5, 5, 5, 5], [2, 0, 2, 1], [3, 4, 3, 4], [3, 5, 2, 0], [1, 3, 5, 2], [5, 5, 5, 5], [4, 4, 2, 1], [5, 3, 5, 5], [3, 5, 3, 5], [1, 3, 1, 4], [1, 5, 1, 1], [

In [128]:
print(f"组1: {grouped_ranks_0[:5]}")
print(f"组2: {grouped_ranks_1[:5]}")

组1: [[3, 1, 4, 2], [3, 2, 4, 1], [2, 3, 1, 4], [4, 1, 2, 3], [1, 2, 3, 4]]
组2: [[3, 1, 4, 2], [3, 2, 4, 1], [2, 3, 1, 4], [4, 1, 2, 3], [1, 2, 3, 4]]


In [78]:
for output in outputs[20:24]:
    print(output.outputs[0].text)
    break

Job Title: Software Engineer

Company: [Company name]

Description:
The Software Engineer is responsible for designing, developing, and maintaining software applications. You will work on various projects and be part of a highly collaborative environment. The ideal candidate should have experience with modern software programming languages, cloud systems, and SQL queries. Strong communication skills and eagerness to work collaboratively are essential for success in this role.

Responsibilities:
- Receive and perform code reviews with other engineers.
- Write unit, integration, and end-to-end tests to verify functionality using automated testing frameworks such as Pytest.
- Work collaboratively with fellow software engineers to build features requested by business stakeholders.
- Participate in Agile teams to develop, test, and debug complex data processing pipelines and data analysis applications using big data processing systems such as Apache Spark.
- Diagnose, debug, and perform roo